# نهج للمتغيرات الفئوية



المتغيرات الفئوية هي جوهر العديد من المهام في العالم الحقيقي. ستتضمن كل مهمة عمل ستحلها على الإطلاق متغيرات فئوية. لذلك من الأفضل أن يكون لديك ذوق جيد منهم.



لأغراض العرض التوضيحي، سأستخدم نموذجين RF وLinear لأن لهما طبيعة مختلفة ومن الأفضل تسليط الضوء على الاختلافات في معالجة الفئات.
مجموعة البيانات من المنافسة المتوسطة kaggle، حيث يجب أن نتوقع عددًا من التصفيقات (الإعجابات) للمقالة.


In [ ]:
import warnings

import feather
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.manifold import TSNE
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

In [ ]:
def add_date_parts(df, date_column="published"):
    df["hour"] = df[date_column].dt.hour
    df["month"] = df[date_column].dt.month
    df["weekday"] = df[date_column].dt.weekday
    df["year"] = df[date_column].dt.year
    df["week"] = df[date_column].dt.week
    df["working_day"] = (df["weekday"] < 5).astype("int")

In [ ]:
PATH_TO_DATA = "../../data/medium/"
train_df = feather.read_dataframe(PATH_TO_DATA + "medium_train")
train_df.set_index("id", inplace=True)
add_date_parts(train_df)

In [ ]:
train_df.head(1)


النص ليس هو الغرض من هذا البرنامج التعليمي، لذلك سأقوم بإسقاطه


In [ ]:
train_df = train_df[
    [
        "author",
        "domain",
        "lang",
        "log_recommends",
        "hour",
        "month",
        "weekday",
        "year",
        "week",
        "working_day",
    ]
]
train_df.head(1)


### النهج الأساسي جنيه.



LE (ترميز التسمية) هو الأكثر بساطة. لدينا بعض الفئات (البلد على سبيل المثال) ["روسيا"، "الولايات المتحدة الأمريكية"، "GB"]. لكن الخوارزميات لا تعمل مع السلاسل، بل تحتاج إلى أرقام. حسنًا، يمكننا القيام بذلك ['روسيا'، 'الولايات المتحدة الأمريكية'، 'GB'] -> [0، 1، 2]. ريلي بسيط. دعونا نحاول.


In [ ]:
autor_to_int = dict(
    (zip(train_df.author.unique(), range(train_df.author.unique().shape[0])))
)
domain_to_int = dict(
    (zip(train_df.domain.unique(), range(train_df.domain.unique().shape[0])))
)
lang_to_int = dict(
    (zip(train_df.lang.unique(), range(train_df.lang.unique().shape[0])))
)
train_df_le = train_df.copy()

In [ ]:
train_df_le["author"] = train_df_le["author"].apply(lambda aut: autor_to_int[aut])
train_df_le["domain"] = train_df_le["domain"].apply(lambda aut: domain_to_int[aut])
train_df_le["lang"] = train_df_le["lang"].apply(lambda aut: lang_to_int[aut])
train_df_le.head()

In [ ]:
y = train_df_le.log_recommends
X = train_df_le.drop("log_recommends", axis=1)


#### تسمية الترددات اللاسلكية المشفرة


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
preds = rf.predict(X_val)
mean_absolute_error(y_val, preds)


#### تسمية LR مشفرة



النماذج الخطية مثل المدخلات المقاسة


In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

In [ ]:
ridge = Ridge()
ridge.fit(X_train, y_train)
preds = ridge.predict(X_val)
mean_absolute_error(y_val, preds)


يبدو أن النموذج الخطي يؤدي أداءً أسوأ. نعم، هذا بسبب طبيعتها. يحاول النموذج الخطي إيجاد الوزن W الذي يمكن ضربه بالمدخل X، y = W*X + b. مع LE نخبر النموذج (مع تعيين ['روسيا'، 'الولايات المتحدة الأمريكية'، 'GB'] -> [0، 1، 2])، هذا الوزن في "روسيا" لا يهم لأن X ==0، وأن جيجا بايت أكبر مرتين من الولايات المتحدة الأمريكية.
لذلك ليس من المناسب استخدام LE مع النماذج الخطية.



### ترميز واحد ساخن (OHE)



يمكننا أن نتعامل مع الفئة باعتبارها الشيء في حد ذاته. سيتم تحويل ['روسيا'، 'الولايات المتحدة الأمريكية'، 'GB'] إلى 3 ميزات، كل منها يأخذ القيمة 0 أو 1.
بهذه الطريقة يمكننا التعامل مع الميزات بشكل مستقل، ولكن العلاقة الأساسية تنفجر.


In [ ]:
train_df_ohe = train_df.copy()
y = train_df_ohe.log_recommends
X = train_df_ohe.drop("log_recommends", axis=1)
X[X.columns] = X[X.columns].astype("category")
X = pd.get_dummies(X, prefix=X.columns)

In [ ]:
X.shape


بوم! لقد كانت 9 أبعاد والآن أصبحت 317 ألف أبعاد. (نعم، أتعامل مع اليوم والسنة والأسبوع كفئة)


#### الترددات اللاسلكية أوه


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)
rf = RandomForestRegressor(n_jobs=-1)
rf.fit(X_train, y_train)
preds = rf.predict(X_val)
mean_absolute_error(y_val, preds)


تحسنت النتيجة ولكن وقت التعلم واستهلاك الذاكرة قفز بشكل كبير. (كان > 20 جيجابايت من ذاكرة الوصول العشوائي)



#### إل آر أوه


In [ ]:
ridge = Ridge()
ridge.fit(X_train, y_train)
preds = ridge.predict(X_val)
mean_absolute_error(y_val, preds)


واو! تحسن كبير.



### التضمينات الفئوية



أنت تعرف بالفعل كل ما هو أعلاه.
الآن حان الوقت لتجربة شيء جديد. سننظر إلى نهج NN للمتغيرات الفئوية.
في مسابقات kaggle، يمكننا أن نرى أنه في المسابقات ذات الاستخدام المكثف لطرق تجميع شجرة البيانات الفئوية تعمل بشكل أفضل (XGBoost). لماذا في عصور صعود NN ما زالوا لم يغزوا هذه المنطقة؟<br>
من حيث المبدأ، يمكن للشبكة العصبية أن تقريب أي وظيفة مستمرة ووظيفة مستمرة متعددة التعريف. ومع ذلك، فهي غير مناسبة لتقريب الوظائف الاعتباطية غير المستمرة لأنها تفترض مستوى معينًا من الاستمرارية في شكلها العام. أثناء مرحلة التدريب، تضمن استمرارية البيانات تقارب التحسين، وأثناء مرحلة التنبؤ تضمن أن التغيير الطفيف في قيم المدخلات يحافظ على استقرار المخرجات.<br>
لا تملك الأشجار هذا الافتراض بشأن استمرارية البيانات ويمكنها تقسيم حالات المتغير بالدقة اللازمة.


NN قريب إلى حد ما من النموذج الخطي. ماذا فعلنا بالنموذج الخطي؟ استخدمنا OHE، لكنه فجر أبعادنا. بالنسبة للعديد من المهام الواقعية، عندما يكون للميزات عدد أساسي من الملايين، سيكون الأمر أكثر صعوبة. ثانيا، لقد فقدنا بعض المعلومات مع مثل هذا التحول. في مثالنا، لدينا اللغة كميزة. عندما نقوم بتحويل "الإسبانية" -> [1,0,0,...,0] وعندما نقوم بالتحويل "الإنجليزية" -> [0,1,0,...,0]. كلتا اللغتين لهما نفس المسافة بين بعضهما البعض، ولكن ليس هناك شك في أن الإسبانية والإنجليزية أكثر تشابهًا من اللغتين الإنجليزية والصينية. نريد الحصول على هذه العلاقة الداخلية.



الحل لهذه المشاكل هو استخدام <b>embeddings</b>، الذي يترجم المتجهات الكبيرة المتفرقة إلى مساحة ذات أبعاد أقل تحافظ على العلاقات الدلالية.



آلية العمل في مجال البرمجة اللغوية العصبية:
| ميزة | ناقلات |
|------|------|
| جرو | [0.9، 1.0، 0.0] |
|   كلب | [1.0، 0.2، 0.0]|
|   هريرة | [0.0، 1.0، 0.9]|
|   قطة | [0.0، 0.2، 1.0]|



نرى أن الكلمات تشترك في بعض القيم التي يمكن أن نعتبرها "الحجم" أو "الحجم".



للقيام بذلك، كل ما نحتاجه هو مصفوفة التضمين.
في البداية، نقوم بتطبيق OHE ونحصل على صفوف N مع أعمدة M. حيث m هي قيمة الفئة. ثم نختار الصف الذي يشفر فئتنا من مصفوفة التضمين. علاوة على ذلك، نستخدم هذا المتجه الذي يمثل بعض الخصائص الغنية لفئتنا الأولية.  
يمكننا الحصول على التضمينات باستخدام سحر NN. نحن نقوم بتدريب مصفوفة التضمين بحجم MxP حيث P هو الرقم الذي نختاره (المعلمة المفرطة). يخبرنا إرشاد Google أن نختار M ** 0.25


In [ ]:
from IPython.display import Image

Image(url="https://habrastorage.org/webt/of/jy/gd/ofjygd5fmbpxwz8x6boeu2nnpk4.png")


سأستخدم keras، لكن ليس المهم أنها مجرد أداة.


In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras.layers import BatchNormalization, Dense, Dropout, Embedding, Input
from keras.models import Model, Sequential

In [ ]:
class EmbeddingMapping:
    """
    Helper class for handling categorical variables
    An instance of this class should be defined for each categorical variable we want to use.
    """

    def __init__(self, series):
        # get a list of unique values
        values = series.unique().tolist()

        # Set a dictionary mapping from values to integer value
        self.embedding_dict = {
            value: int_value + 1 for int_value, value in enumerate(values)
        }

        # The num_values will be used as the input_dim when defining the embedding layer.
        # It will also be returned for unseen values
        self.num_values = len(values) + 1

    def get_mapping(self, value):
        # If the value was seen in the training set, return its integer mapping
        if value in self.embedding_dict:
            return self.embedding_dict[value]
        # Else, return the same integer for unseen values
        else:
            return self.num_values

In [ ]:
# converting some out features
author_mapping = EmbeddingMapping(train_df["author"])
domain_mapping = EmbeddingMapping(train_df["domain"])
lang_mapping = EmbeddingMapping(train_df["lang"])
X_emb = X_emb.assign(author_mapping=X_emb["author"].apply(author_mapping.get_mapping))
X_emb = X_emb.assign(lang_mapping=X_emb["lang"].apply(lang_mapping.get_mapping))
X_emb = X_emb.assign(domain_mapping=X_emb["domain"].apply(domain_mapping.get_mapping))

In [ ]:
X_emb.sample(1)

In [ ]:
X_emb = train_df.copy()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_emb, y, test_size=0.2)

In [ ]:
# Keras functional API
# Input
author_input = Input(shape=(1,), dtype="int32")
lang_input = Input(shape=(1,), dtype="int32")
domain_input = Input(shape=(1,), dtype="int32")

# It's google's fule of thumb N_embeddings == N_originall_dim**0.25
# Let’s define the embedding layer and flatten it
# Originally 31331 unique authors
author_embedings = Embedding(
    output_dim=13, input_dim=author_mapping.num_values, input_length=1
)(author_input)
author_embedings = keras.layers.Reshape((13,))(author_embedings)
# Originally 62 unique langs
lang_embedings = Embedding(
    output_dim=3, input_dim=lang_mapping.num_values, input_length=1
)(lang_input)
lang_embedings = keras.layers.Reshape((3,))(lang_embedings)
# Originally 221 unique domains
domain_embedings = Embedding(
    output_dim=4, input_dim=domain_mapping.num_values, input_length=1
)(domain_input)
domain_embedings = keras.layers.Reshape((4,))(domain_embedings)


# Concatenate continuous and embeddings inputs
all_input = keras.layers.concatenate(
    [lang_embedings, author_embedings, domain_embedings]
)

In [ ]:
# Fully connected layer to train NN and learn embeddings
units = 25
dense1 = Dense(units=units, activation="relu")(all_input)
dense1 = Dropout(0.5)(dense1)
dense2 = Dense(units, activation="relu")(dense1)
dense2 = Dropout(0.5)(dense2)
predictions = Dense(1)(dense2)

In [ ]:
epochs = 40
model = Model(inputs=[lang_input, author_input, domain_input], outputs=predictions)
model.compile(loss="mae", optimizer="adagrad")

history = model.fit(
    [X_train["lang_mapping"], X_train["author_mapping"], X_train["domain_mapping"]],
    y_train,
    epochs=epochs,
    batch_size=128,
    verbose=0,
    validation_data=(
        [X_val["lang_mapping"], X_val["author_mapping"], X_val["domain_mapping"]],
        y_val,
    ),
)

في هذه الخطوة، قمنا بتدريب NN، لكننا لن نستخدمها. نريد الحصول على طبقة التضمين.
لكل فئة، لدينا تضمين مميز. دعونا نستخرجها ونستخدمها في نماذجنا البسيطة.


In [ ]:
model.layers

In [ ]:
model.layers[5].get_weights()[0].shape

In [ ]:
lang_embedding = model.layers[3].get_weights()[0]
lang_emb_cols = [f"lang_emb_{i}" for i in range(lang_embedding.shape[1])]

In [ ]:
author_embedding = model.layers[4].get_weights()[0]
aut_emb_cols = [f"aut_emb_{i}" for i in range(author_embedding.shape[1])]

In [ ]:
domain_embedding = model.layers[5].get_weights()[0]
dom_emb_cols = [f"dom_emb_{i}" for i in range(domain_embedding.shape[1])]


الآن لدينا التضمينات، وكل ما نحتاجه هو أن نأخذ صفًا يتوافق مع الأمثلة التي لدينا.


In [ ]:
def get_author_vector(aut_num):
    return author_embedding[aut_num, :]


def get_lang_vector(lang_num):
    return lang_embedding[lang_num, :]


def get_domain_vector(dom_num):
    return domain_embedding[dom_num, :]

In [ ]:
get_lang_vector(4)

In [ ]:
lang_emb = pd.DataFrame(
    X_emb["lang_mapping"].apply(get_lang_vector).values.tolist(), columns=lang_emb_cols
)
lang_emb.index = X_emb.index
X_emb[lang_emb_cols] = lang_emb

In [ ]:
aut_emb = pd.DataFrame(
    X_emb["author_mapping"].apply(get_author_vector).values.tolist(),
    columns=aut_emb_cols,
)
aut_emb.index = X_emb.index
X_emb[aut_emb_cols] = aut_emb

In [ ]:
dom_emb = pd.DataFrame(
    X_emb["domain_mapping"].apply(get_domain_vector).values.tolist(),
    columns=dom_emb_cols,
)
dom_emb.index = X_emb.index
X_emb[dom_emb_cols] = dom_emb

In [ ]:
X_emb.drop(
    [
        "author",
        "lang",
        "domain",
        "log_recommends",
        "author_mapping",
        "lang_mapping",
        "domain_mapping",
    ],
    axis=1,
    inplace=True,
)

In [ ]:
X_emb.columns

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_emb, y, test_size=0.2)

In [ ]:
rf = RandomForestRegressor(n_jobs=-1)
rf.fit(X_train, y_train)
preds = rf.predict(X_val)
mean_absolute_error(y_val, preds)

In [ ]:
ridge = Ridge()
ridge.fit(X_train, y_train)
preds = ridge.predict(X_val)
mean_absolute_error(y_val, preds)


يبدو وكأنه نجاح.



إحدى الخصائص الرائعة للتضمين - فئاتنا لديها بعض التشابه (المسافة) عن بعضها البعض. دعونا ننظر إلى الرسم البياني.


In [ ]:
import bokeh.models as bm
import bokeh.plotting as pl
from bokeh.io import output_notebook

output_notebook()


def draw_vectors(
    x,
    y,
    radius=10,
    alpha=0.25,
    color="blue",
    width=600,
    height=400,
    show=True,
    **kwargs,
):
    """ draws an interactive plot for data points with auxilirary info on hover """
    if isinstance(color, str):
        color = [color] * len(x)
    data_source = bm.ColumnDataSource({"x": x, "y": y, "color": color, **kwargs})

    fig = pl.figure(active_scroll="wheel_zoom", width=width, height=height)
    fig.scatter("x", "y", size=radius, color="color", alpha=alpha, source=data_source)

    fig.add_tools(bm.HoverTool(tooltips=[(key, "@" + key) for key in kwargs.keys()]))
    if show:
        pl.show(fig)
    return fig

In [ ]:
langs_vectors = [get_lang_vector(l) for l in lang_mapping.embedding_dict.values()]

In [ ]:
lang_tsne = TSNE().fit_transform(langs_vectors)

In [ ]:
draw_vectors(
    lang_tsne[:, 0], lang_tsne[:, 1], token=list(lang_mapping.embedding_dict.keys())
)

In [ ]:
langs_vectors_pca = PCA(n_components=2).fit_transform(langs_vectors)

In [ ]:
draw_vectors(
    langs_vectors_pca[:, 0],
    langs_vectors_pca[:, 1],
    token=list(lang_mapping.embedding_dict.keys()),
)


هذه المرة الرسوم البيانية لا تبدو ذات معنى، ولكن النتيجة تتحدث عن نفسها.



## كات2فيك



هناك طريقة أخرى جاءت من البرمجة اللغوية العصبية وهي word2Vec والتي تمت إعادة تسميتها إلى Cat2Vec. ليس هناك تأكيد قاطع حول فائدته، ولكن هناك بعض الأوراق التي تجادل بذلك. (الروابط أدناه).



دلالات التوزيع ويقول جون روبرت فيرث "يجب أن تعرف الكلمة من خلال الشركة التي تحتفظ بها". بعض الكلمات تشترك في نفس السياق، لذلك فهي متشابهة إلى حد ما. يمكننا أن نقترح أن الفئات قد تشترك في بعض الارتباطات الداخلية من خلال حدوثها بشكل مشترك. على سبيل المثال الطقس والمدينة. ربما تكون مدينة "فيلادلفيا" مشابهة للطقس "مشمس دائمًا"، أو "موسكو" مع "مثلج".



أولاً، نقوم بتطبيق تشفير الميزات، ثم يمكننا إنشاء "جملة" من صفنا.
في المثال أدناه، لنتخيل أن لدينا مقالًا على "Monday January 2018 English_language Medium.com" هنا الجملة الخاصة بنا، لذلك ربما إذا كانت اللغة الإنجليزية تتزامن مع اللغة المتوسطة في كثير من الأحيان ثم اللغة الصينية مع hackernoon.com. (سوء الاعتبار ولكن فقط على سبيل المثال).



الاعتبار الوحيد هو ترتيب "الكلمات". يقوم Word2Vec بالترحيل حسب الترتيب، ولا يهم "الجملة" الفئوية، لذا من الأفضل خلط الجمل عشوائيًا.
دعونا ننفذها.


In [ ]:
X_w2v = train_df.copy()

In [ ]:
month_int_to_name = {
    1: "jan",
    2: "feb",
    3: "apr",
    4: "march",
    5: "may",
    6: "june",
    7: "jul",
    8: "aug",
    9: "sept",
    10: "okt",
    11: "nov",
    12: "dec",
}
weekday_int_to_day = {
    0: "mon",
    1: "thus",
    2: "wen",
    3: "thusd",
    4: "fri",
    5: "sut",
    6: "sun",
}

In [ ]:
working_day_int_to_day = {1: "work", 0: "not_work"}

In [ ]:
X_w2v.month = X_w2v.month.apply(lambda x: month_int_to_name[x])

In [ ]:
X_w2v.weekday = X_w2v.weekday.apply(lambda x: weekday_int_to_day[x])

In [ ]:
X_w2v.working_day = X_w2v.working_day.apply(lambda x: working_day_int_to_day[x])

In [ ]:
all_list = list()
for ind, r in X_w2v.iterrows():
    values_list = [str(val).replace(" ", "_") for val in r.values]
    all_list.append(values_list)

In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(
    all_list,
    size=32,  # embedding vector size
    min_count=5,  # consider words that occured at least 5 times
    window=5,
).wv

In [ ]:
model.most_similar("june")

In [ ]:
words = sorted(
    model.vocab.keys(), key=lambda word: model.vocab[word].count, reverse=True
)[:1000]

print(words[::100])

In [ ]:
word_vectors = np.array([model.get_vector(wrd) for wrd in words])


ارسم رسمًا بيانيًا كالمعتاد


In [ ]:
word_tsne = TSNE().fit_transform(word_vectors)

In [ ]:
draw_vectors(word_tsne[:, 0], word_tsne[:, 1], color="green", token=words)

لقد اختلطت فئاتنا، لكن يمكننا أن نلاحظ أن السنوات والأيام واللغات تظل بعيدة عن سحابة المؤلفين.


In [ ]:
def get_phrase_embedding(phrase):
    """
    Convert phrase to a vector by aggregating it's word embeddings. See description above.
    """
    # 1. lowercase phrase
    # 2. tokenize phrase
    # 3. average word vectors for all words in tokenized phrase
    # skip words that are not in model's vocabulary
    # if all words are missing from vocabulary, return zeros

    vector = np.zeros([model.vector_size], dtype="float32")
    word_count = 0

    for word in phrase.split():
        if word in model.vocab:
            vector += model.get_vector(word)
            word_count += 1

    if word_count:
        vector /= word_count

    return vector

In [ ]:
new_features = list()
for ph in all_list:
    vector = get_phrase_embedding(" ".join(ph))
    new_features.append(vector)

In [ ]:
new_features = pd.DataFrame(new_features)
new_features.index = X_w2v.index
X_w2v = pd.concat([X_w2v, new_features], axis=1)

In [ ]:
X_w2v.drop(
    [
        "author",
        "domain",
        "lang",
        "working_day",
        "year",
        "month",
        "weekday",
        "log_recommends",
    ],
    axis=1,
    inplace=True,
)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_w2v, y, test_size=0.2)

In [ ]:
rf = RandomForestRegressor(n_jobs=-1)
rf.fit(X_train, y_train)
preds = rf.predict(X_val)
mean_absolute_error(y_val, preds)

In [ ]:
ridge = Ridge()
ridge.fit(X_train, y_train)
preds = ridge.predict(X_val)
mean_absolute_error(y_val, preds)


نتيجة سيئة، لكنني قمت بقص الكثير من الميزات التي يمكن أن تساعد هذه الخوارزمية في صياغة الكلمات.



## الاستنتاجات



الآن أنت تعرف أن المتغيرات الفئوية هي وحش صعب وأنه يمكننا الحصول على الكثير منها عن طريق التضمين وتقنيات cat2Vec. إنها لا تعمل فقط مع NN ولكن في نماذج أبسط، لذلك من الممكن استخدامها في أنظمة الإنتاج ذات زمن الوصول المنخفض.



https://arxiv.org/ftp/arxiv/papers/1603/1603.04259.pdf ITEM2VEC: تضمين العنصر العصبي للتصفية التعاونية<br>
https://openreview.net/pdf?id=HyNxRZ9xg CAT2VEC: تعلم التمثيل الموزع للبيانات الفئوية متعددة الحقول<br>
https://arxiv.org/pdf/1604.06737v1.pdf تضمينات الكيان للمتغيرات الفئوية<br>
https://developers.google.com/machine-learning/crash-course/embeddings/video-lecture التضمينات<br>